# Notebook 04 - Model Evaluation & Metrics
## Human Intrusion Detection System
Evaluates trained YOLOv8s: mAP, speed, zone logic, tracker.
**GPU**: NVIDIA Quadro T2000 | **Classes**: person only


In [ ]:
# Fix: Robust root detection (works from any Jupyter launch directory)
import sys
from pathlib import Path
def _find_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / 'src').is_dir() and (p / 'requirements.txt').exists():
            return p
    return Path.cwd()
ROOT = _find_root()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')
print(f'src/ found  : {(ROOT / "src").is_dir()}')

import time, warnings, json
from collections import defaultdict
import torch, numpy as np, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns, pandas as pd
from ultralytics import YOLO
from shapely.geometry import Polygon, Point

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='darkgrid')

PROC_DIR   = ROOT / 'data' / 'processed'
WEIGHTS    = ROOT / 'weights'
ZONES_FILE = ROOT / 'data' / 'zones' / 'zones_config.json'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

best_weights = WEIGHTS / 'yolov8s_intrusion.pt'
if best_weights.exists():
    model = YOLO(str(best_weights))
    print(f'Model loaded: {best_weights.name}')
else:
    print('Trained weights not found - loading pretrained yolov8s for demo.')
    model = YOLO('yolov8s.pt')


## Section 1 - mAP Validation

In [ ]:
yaml_path = PROC_DIR / 'intrusion.yaml'
if yaml_path.exists():
    print('Running validation...')
    metrics = model.val(data=str(yaml_path), imgsz=640, batch=8,
                        device=DEVICE, half=True, conf=0.001, iou=0.6, verbose=False, plots=False)
    P, R = metrics.box.mp, metrics.box.mr
    F1 = 2 * P * R / (P + R + 1e-8)
    print(f'mAP@0.5    : {metrics.box.map50:.4f}')
    print(f'mAP@0.5:95 : {metrics.box.map:.4f}')
    print(f'Precision  : {P:.4f}')
    print(f'Recall     : {R:.4f}')
    print(f'F1 Score   : {F1:.4f}')
    if metrics.box.map50 >= 0.85: print('EXCELLENT - exceeds target mAP >= 0.85')
    elif metrics.box.map50 >= 0.75: print('GOOD - consider more epochs')
    else: print('Below target - check dataset and config')
else:
    print('Dataset YAML not found. Run Notebooks 01-03 first.')


## Section 2 - Speed Benchmark

In [ ]:
import time
print('Speed Benchmark - NVIDIA Quadro T2000')
results_bench = []
for W, H in [(640, 480), (1280, 720), (1920, 1080)]:
    dummy = __import__('numpy').random.randint(0,255,(H,W,3),dtype=__import__('numpy').uint8)
    for _ in range(3): model.predict(dummy, device=DEVICE, conf=0.5, verbose=False, half=True)
    times = []
    for _ in range(30):
        t0 = time.perf_counter()
        model.predict(dummy, device=DEVICE, conf=0.5, imgsz=640, verbose=False, half=True)
        times.append((time.perf_counter()-t0)*1000)
    avg = __import__('numpy').mean(times); fps = 1000/avg
    ok = 'PASS' if fps >= 15 else 'FAIL'
    print(f'  {W}x{H}: {avg:.1f}ms | {fps:.1f} FPS | Target>=15FPS: {ok}')
    results_bench.append({'Resolution':f'{W}x{H}','FPS':f'{fps:.1f}','Target':ok})
print('Done.')


## Section 3 - Zone Logic Test

In [ ]:
from src.inference.zone_manager import ZoneManager
from src.models.detector import Detection

if ZONES_FILE.exists():
    zm = ZoneManager(config_path=str(ZONES_FILE), cooldown_seconds=0)
    print('Zone Logic Test')
    test_cases = [
        {'cx':300,'cy':400,'cam':'CAM_01','expected':True, 'desc':'Person inside Storage Area'},
        {'cx':700,'cy':200,'cam':'CAM_01','expected':False,'desc':'Person outside all zones'},
        {'cx':200,'cy':200,'cam':'CAM_02','expected':True, 'desc':'Person inside Server Room'},
    ]
    passed = 0
    for tc in test_cases:
        cx, cy, half = tc['cx'], tc['cy'], 20
        det = Detection(bbox=(cx-half,cy-half,cx+half,cy+half),
                        confidence=0.85, class_id=0, class_name='person')
        alerts = zm.evaluate([det], camera_id=tc['cam'])
        got = len(alerts) > 0
        ok = got == tc['expected']
        if ok: passed += 1
        status = 'PASS' if ok else 'FAIL'
        print(f'  [{status}] {tc["desc"]} | expected={tc["expected"]} got={got}')
    print(f'Zone Accuracy: {passed}/{len(test_cases)}')
else:
    print('zones_config.json not found.')


## Section 4 - ByteTrack Test

In [ ]:
from src.models.tracker import ByteTracker
from src.models.detector import FrameResult, Detection

print('ByteTrack Integration Test')
tracker = ByteTracker(track_thresh=0.5, min_hits=2, track_buffer=30)
track_ids_seen = set()
for frame_num in range(10):
    cx = 100 + frame_num * 30
    det = Detection(bbox=(cx-40,260,cx+40,340), confidence=0.85,
                    class_id=0, class_name='person')
    dets = [det]
    if frame_num >= 5:
        dets.append(Detection(bbox=(500,200,560,320), confidence=0.78,
                              class_id=0, class_name='person'))
    fr = FrameResult(frame_id=frame_num, timestamp=frame_num/15.0,
                     camera_id='CAM_01', detections=dets)
    updated = tracker.update(fr)
    ids = [d.track_id for d in updated.detections if d.track_id]
    track_ids_seen.update(ids)
    print(f'  Frame {frame_num:02d}: dets={len(dets)} ids={ids}')
print(f'Unique track IDs: {len(track_ids_seen)} (expected 2)')
